In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix, vstack
from scipy.stats import zscore

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/atac/peaks/clusters/justin_chromhmm_cluster_donor.h5ad')

In [ ]:
adata

In [ ]:
adata.X

In [ ]:
tmp_mtx = pd.DataFrame(adata.X)
cpm = tmp_mtx.div(tmp_mtx.sum(axis=1), axis=0) * 1e6
norm_zscore = cpm.apply(zscore).fillna(0)
norm_zscore.shape

In [ ]:
norm_zscore.mean(axis=0)

In [ ]:
adata.X = norm_zscore.values

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

In [ ]:
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
tmp_clusts = ['Start_1', 'Start_2',
                   'Sendai_1', 'Sendai_2',
                   'Delayed_1', 'Delayed_2', 'Delayed_3',
                  'Fail1_1', 'Fail1_2', 'Fail1_3', 'Fail1_4',
                  'Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
                  'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
                  'Fail2_2', 'Fail2_3', 'Fail2_4', 'Fail2_5',
                    'IPS_1', 'IPS_2', 'IPS_3',]

In [ ]:
annots = peaks[14].unique()
annots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(40, 14))

chromhmm_results = []
for i,ax in enumerate(axes.flatten()):
    a = annots[i]
    print(a)
    annot_peaks = peaks[peaks[14] == a]
    peak_results = []
    for s in tmp_clusts:
        tmp_peaks = annot_peaks[annot_peaks[12] == s]
        mask = adata.var.index.isin(tmp_peaks.index)
        tmp_adata = adata[:,mask]
        peak_results.append(tmp_adata.X.mean(axis=1))

    peak_df = pd.DataFrame(np.vstack(peak_results))
    peak_df.columns = clusts
    peak_df.index = tmp_clusts
    peak_df = peak_df.dropna()

    sns.heatmap(peak_df, vmin=-2, vmax=2, cmap='coolwarm', ax=ax)
    ax.set_title(a)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=6)
    ax.grid(False)
    
    peak_df['peak_type'] = a
    peak_df['sub_clust'] = peak_df.index
    chromhmm_results.append(peak_df)
    
plt.subplots_adjust(wspace=0.2, hspace=0.4)
plt.tight_layout()

In [ ]:
final_merged = pd.concat(chromhmm_results)
final_merged.head()

In [ ]:
# #     clust_results.append(tmp_adata.X)
    
# merged.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
final_merged.to_csv(f'{outdir}/figure3/atac_chromhmm_peaks_zscore_cluster_donor.csv', sep='\t', index=None)

### do it for just cluster


In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/atac/peaks/clusters/justin_chromhmm_cluster.h5ad')
tmp_mtx = pd.DataFrame(adata.X)
cpm = tmp_mtx.div(tmp_mtx.sum(axis=1), axis=0) * 1e6
norm_zscore = cpm.apply(zscore).fillna(0)
norm_zscore.shape
adata.X = norm_zscore.values
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(40, 14), dpi=300)

chromhmm_results = []
for i,ax in enumerate(axes.flatten()):
    a = annots[i]
    print(a)
    annot_peaks = peaks[peaks[14] == a]
    peak_results = []
    for s in tmp_clusts:
        tmp_peaks = annot_peaks[annot_peaks[12] == s]
        mask = adata.var.index.isin(tmp_peaks.index)
        tmp_adata = adata[:,mask]
        peak_results.append(tmp_adata.X.mean(axis=1))

    peak_df = pd.DataFrame(np.vstack(peak_results))
    peak_df.columns = tmp_order
    peak_df.index = tmp_clusts
    peak_df = peak_df.dropna()

    sns.heatmap(peak_df, vmin=-2, vmax=2, cmap='coolwarm', ax=ax)
    ax.set_title(a)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=6)
    ax.grid(False)
    
    peak_df['peak_type'] = a
    peak_df['sub_clust'] = peak_df.index
    chromhmm_results.append(peak_df)
    
plt.subplots_adjust(wspace=0.2, hspace=0.4)
plt.tight_layout()

In [ ]:
final_merged = pd.concat(chromhmm_results)
final_merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
final_merged.to_csv(f'{outdir}/figure3/atac_chromhmm_peaks_zscore_cluster.csv', sep='\t', index=None)

In [ ]:
!date